# 🧬 Introduction

This project explores the impact of various **Genetic Algorithm (GA)** operators on optimization performance. Genetic Algorithms are heuristic search methods inspired by the principles of natural selection and evolution, commonly applied to solve complex optimization problems.

To assess the effectiveness of different GA components, we compared multiple:
- **Selection strategies**: Tournament, Rank, and Roulette Wheel.
- **Crossover methods**: Arithmetic, Uniform, and Single-Point.
- **Mutation techniques**: Gaussian and Uniform.

The performance of each combination was evaluated on two well-known benchmark functions:
1. **Sphere Function** – a simple, convex, and continuous test case for general convergence behavior.
2. **Rosenbrock Function** – a more complex, non-convex problem used to test the algorithm's robustness and precision.

The goal was to determine which operator configurations yield the best performance across different landscapes, offering insights into effective GA design and tuning.


In [30]:
import math
import random
import numpy as np
import plotly.graph_objects as go
from typing import Callable, List, Tuple, Dict

# Core functions
def sphere_function(x: np.ndarray) -> float:
    """Sphere function (sum of squares) with optimum at 0."""
    return np.sum(x**2)

def rosenbrock_function(x: np.ndarray) -> float:
    """Rosenbrock function with optimum at 1."""
    return np.sum(100.0 * (x[1:] - x[:-1]**2)**2 + (1 - x[:-1])**2)

def fitness_function(population: np.ndarray, func: Callable, optimum: float = 0.0) -> np.ndarray:
    """Compute absolute distance from optimum for each candidate."""
    return np.abs(np.array([func(individual) for individual in population]) - optimum)

# ====================== CROSSOVER OPERATORS ======================
def arithmetic_crossover(parent1: np.ndarray, parent2: np.ndarray) -> np.ndarray:
    """Arithmetic crossover (averaging)."""
    return (parent1 + parent2) / 2

def single_point_crossover(parent1: np.ndarray, parent2: np.ndarray) -> np.ndarray:
    """Single-point crossover."""
    if len(parent1) != len(parent2):
        raise ValueError("Parents must have the same dimension")
    crossover_point = random.randint(1, len(parent1) - 1)
    return np.concatenate([parent1[:crossover_point], parent2[crossover_point:]])

def uniform_crossover(parent1: np.ndarray, parent2: np.ndarray) -> np.ndarray:
    """Uniform crossover with 50% mixing ratio."""
    mask = np.random.randint(0, 2, size=len(parent1))
    return parent1 * mask + parent2 * (1 - mask)

# ====================== MUTATION OPERATORS ======================
def gaussian_mutation(candidate: np.ndarray, mutation_rate: float = 0.05, sigma: float = 0.1) -> np.ndarray:
    mask = np.random.random(len(candidate)) < mutation_rate
    noise = np.random.normal(0, sigma, len(candidate))
    return candidate + (mask * noise)

def uniform_mutation(candidate: np.ndarray, mutation_rate: float = 0.05, mutation_range: float = 0.1) -> np.ndarray:
    """Uniform random mutation."""
    mask = np.random.random(len(candidate)) < mutation_rate
    noise = np.random.uniform(-mutation_range, mutation_range, len(candidate))
    return candidate + (mask * noise)

# ====================== SELECTION OPERATORS ======================
def tournament_selection(population: np.ndarray, fitness: np.ndarray, tournament_size: int = 3) -> Tuple[int, int]:
    """Tournament selection with replacement."""
    # Select two separate tournaments for two parents
    candidates1 = np.random.choice(len(fitness), size=tournament_size, replace=False)
    candidates2 = np.random.choice(len(fitness), size=tournament_size, replace=False)
    winner1 = candidates1[np.argmin(fitness[candidates1])]
    winner2 = candidates2[np.argmin(fitness[candidates2])]
    return winner1, winner2

def roulette_wheel_selection(population: np.ndarray, fitness: np.ndarray) -> Tuple[int, int]:
    """Roulette wheel selection based on inverse fitness."""
    inverse_fitness = 1 / (fitness + 1e-10)  # Add small value to avoid division by zero
    probabilities = inverse_fitness / np.sum(inverse_fitness)
    return tuple(np.random.choice(len(fitness), size=2, p=probabilities, replace=True))

def rank_selection(population: np.ndarray, fitness: np.ndarray) -> Tuple[int, int]:
    """Rank-based selection."""
    ranks = np.argsort(np.argsort(fitness))  # Convert fitness to ranks
    probabilities = (len(fitness) - ranks) / np.sum(len(fitness) - ranks)
    return tuple(np.random.choice(len(fitness), size=2, p=probabilities, replace=True))



# `generate_next_generation` Function Documentation

## Overview
This function creates a new population for the next generation of the genetic algorithm using selection, crossover, and mutation operators while optionally preserving elite individuals.

## Parameters

| Parameter | Type | Description | Default |
|-----------|------|-------------|---------|
| `population` | `np.ndarray` | Current population of candidate solutions | - |
| `func` | `Callable` | Objective function being optimized | - |
| `optimum` | `float` | Target optimal value | - |
| `selection_function` | `Callable` | Function to select parent solutions | - |
| `crossover_function` | `Callable` | Function to combine parent solutions | - |
| `mutation_function` | `Callable` | Function to mutate offspring | - |
| `mutation_rate` | `float` | Probability of mutation occurring | `0.05` |
| `mutation_param` | `float` | Magnitude/range parameter for mutation | `0.1` |
| `elitism` | `bool` | Whether to preserve top individuals | `True` |

## Functionality

### 1. Fitness Evaluation
```python
fitness = fitness_function(population, func, optimum)

In [31]:
def generate_next_generation(
    population: np.ndarray,
    func: Callable,
    optimum: float,
    selection_function: Callable,
    crossover_function: Callable,
    mutation_function: Callable,
    mutation_rate: float = 0.05,
    mutation_param: float = 0.1,
    elitism: bool = True
) -> np.ndarray:
    """Generate next generation using specified operators."""
    fitness = fitness_function(population, func, optimum)

    # Sort population by fitness (best first)
    sorted_indices = np.argsort(fitness)
    population = population[sorted_indices]
    fitness = fitness[sorted_indices]

    next_generation = []

    # Elitism: keep the best individual
    if elitism:
        for i in range(0, len(population) // 10):
            next_generation.append(population[i])

    while len(next_generation) < len(population):
        # Select parents
        parent1_idx, parent2_idx = selection_function(population, fitness)
        parent1, parent2 = population[parent1_idx], population[parent2_idx]

        # Crossover
        child = crossover_function(parent1, parent2)

        # Mutate
        child = mutation_function(child, mutation_rate, mutation_param)

        next_generation.append(child)

    return np.array(next_generation)


# Genetic Algorithm Implementation

## Overview
A complete genetic algorithm implementation that evolves solutions over generations. The algorithm maintains a population of candidate solutions and iteratively improves them through selection, crossover, and mutation operations.

## Core Components

1. **Initialization**:
   - Creates random population within `[population_low, population_high]` bounds
   - Tracks best solution and fitness history

2. **Evolution Cycle**:
   - Generates new population each iteration using:
     - Selection (`tournament_selection` default)
     - Crossover (`arithmetic_crossover` default)
     - Mutation (`gaussian_mutation` default)
   - Optional elitism preserves top solutions

3. **Progress Tracking**:
   - Records best solution found
   - Tracks fitness at specified generations
   - Returns: (best_solution, best_fitness, fitness_history)

## Key Features
- Configurable evolutionary operators
- Flexible for different problem dimensions
- Tracks convergence progress
- Supports both minimization and target-reaching

## Default Parameters
- Population size: 100
- Max generations: 100
- Mutation rate: 5%
- Elitism: Enabled

In [32]:
def genetic_algorithm(
    func: Callable,
    optimum: float,
    population_low: float,
    population_high: float,
    population_size: int,
    dimension: int,
    max_generations: int = 100,
    selection_function: Callable = tournament_selection,
    crossover_function: Callable = arithmetic_crossover,
    mutation_function: Callable = gaussian_mutation,
    mutation_rate: float = 0.05,
    mutation_param: float = 0.1,
    elitism: bool = True,
    generations_to_track: List[int] = None
) -> Tuple[np.ndarray, float, List[float]]:
    """Run genetic algorithm optimization with configurable operators."""
    population = np.random.uniform(
        low=population_low,
        high=population_high,
        size=(population_size, dimension)  # Changed to (population_size, dimension)
    )

    best_solution = None
    best_distance = math.inf
    tracked_distances = []

    if generations_to_track is None:
        generations_to_track = list(range(1, max_generations+1))

    for generation in range(max_generations):
        population = generate_next_generation(
            population, func, optimum, selection_function,
            crossover_function, mutation_function,
            mutation_rate, mutation_param, elitism
        )

        # Track progress
        current_fitness = fitness_function(population, func, optimum)
        best_idx = np.argmin(current_fitness)
        new_best_solution = population[best_idx]  # Changed to row index
        new_best_distance = current_fitness[best_idx]

        if new_best_distance < best_distance:
            best_solution = new_best_solution
            best_distance = new_best_distance

        if (generation + 1) in generations_to_track:
            tracked_distances.append(best_distance)

    return best_solution, best_distance, tracked_distances

# GA Operator Comparison

Tests different genetic algorithm components on the sphere function:

**Parameters**:
- Population: 50 solutions, 10D
- Bounds: [-5,5]
- Generations: 300
- Mutation rate: 10%
- Elitism: On

**Comparisons**:
1. **Selection Methods**:
   - Tournament
   - Roulette Wheel
   - Rank

2. **Crossover Types**:
   - Arithmetic
   - Single-Point
   - Uniform

3. **Mutation Types**:
   - Gaussian (σ=0.1)
   - Uniform (±0.1)

**Output**:
- Interactive convergence plots
- Final fitness values
- Log-scaled fitness axis


In [33]:
def run_comparison_experiments():
    """Run comparison experiments for different operators."""
    # Common parameters for all experiments
    maxgen = 300
    common_params = {
        "func": sphere_function,
        "optimum": 0,
        "population_low": -5,
        "population_high": 5,
        "population_size": 50,
        "dimension": 10,
        "max_generations": maxgen,
        "generations_to_track": list(range(1, maxgen+1)),
        "elitism": True
    }

    # ====================== SELECTION COMPARISON ======================
    selection_results = []
    selection_strategies = [
        {"function": tournament_selection, "label": "Tournament Selection"},
        {"function": roulette_wheel_selection, "label": "Roulette Wheel Selection"},
        {"function": rank_selection, "label": "Rank Selection"}
    ]

    for strategy in selection_strategies:
        best_sol, best_dist, track_dist = genetic_algorithm(
            **common_params,
            selection_function=strategy["function"],
            crossover_function=arithmetic_crossover,
            mutation_function=gaussian_mutation,
            mutation_rate=0.1,
            mutation_param=0.1
        )
        selection_results.append({
            "label": strategy["label"],
            "distances": track_dist,
            "final_distance": best_dist
        })

    # ====================== CROSSOVER COMPARISON ======================
    crossover_results = []
    crossover_operators = [
        {"function": arithmetic_crossover, "label": "Arithmetic Crossover"},
        {"function": single_point_crossover, "label": "Single-Point Crossover"},
        {"function": uniform_crossover, "label": "Uniform Crossover"}
    ]

    for operator in crossover_operators:
        best_sol, best_dist, track_dist = genetic_algorithm(
            **common_params,
            selection_function=tournament_selection,
            crossover_function=operator["function"],
            mutation_function=gaussian_mutation,
            mutation_rate=0.1,
            mutation_param=0.1
        )
        crossover_results.append({
            "label": operator["label"],
            "distances": track_dist,
            "final_distance": best_dist
        })

    # ====================== MUTATION COMPARISON ======================
    mutation_results = []
    mutation_operators = [
        {"function": gaussian_mutation, "param": 0.1, "label": "Gaussian Mutation (σ=0.1)"},
        {"function": uniform_mutation, "param": 0.1, "label": "Uniform Mutation (±0.1)"}
    ]

    for operator in mutation_operators:
        best_sol, best_dist, track_dist = genetic_algorithm(
            **common_params,
            selection_function=tournament_selection,
            crossover_function=arithmetic_crossover,
            mutation_function=operator["function"],
            mutation_rate=0.1,
            mutation_param=operator["param"]
        )
        mutation_results.append({
            "label": operator["label"],
            "distances": track_dist,
            "final_distance": best_dist
        })

    # ====================== PLOT RESULTS ======================
    def plot_results(results, title):
        fig = go.Figure()
        for result in results:
            fig.add_trace(go.Scatter(
                x=list(range(1, maxgen + 1)),
                y=result["distances"],
                name=f"{result['label']} (Final: {result['final_distance']:.2e})",
                mode="lines"
            ))
        fig.update_layout(
            title=title,
            xaxis_title="Generation",
            yaxis_title="Distance from Optimum (Log Scale)",
            yaxis_type="log",
            hovermode="x unified"
        )
        fig.show()

    plot_results(selection_results, "Selection Strategy Comparison")
    plot_results(crossover_results, "Crossover Operator Comparison")
    plot_results(mutation_results, "Mutation Operator Comparison")

# Run the comparison experiments
run_comparison_experiments()

# GA Operator Comparison

Tests different genetic algorithm components on the Rosenbrock function:


In [34]:
def run_comparison_experiments():
    """Run comparison experiments for different operators."""
    # Common parameters for all experiments
    maxgen = 300
    common_params = {
        "func": rosenbrock_function,
        "optimum": 0,
        "population_low": -2.048,
        "population_high": 2.048,
        "population_size": 50,
        "dimension": 10,
        "max_generations": maxgen,
        "generations_to_track": list(range(1, maxgen+1)),
        "elitism": True
    }

    # ====================== SELECTION COMPARISON ======================
    selection_results = []
    selection_strategies = [
        {"function": tournament_selection, "label": "Tournament Selection"},
        {"function": roulette_wheel_selection, "label": "Roulette Wheel Selection"},
        {"function": rank_selection, "label": "Rank Selection"}
    ]

    for strategy in selection_strategies:
        best_sol, best_dist, track_dist = genetic_algorithm(
            **common_params,
            selection_function=strategy["function"],
            crossover_function=arithmetic_crossover,
            mutation_function=gaussian_mutation,
            mutation_rate=0.1,
            mutation_param=0.1
        )
        selection_results.append({
            "label": strategy["label"],
            "distances": track_dist,
            "final_distance": best_dist
        })

    # ====================== CROSSOVER COMPARISON ======================
    crossover_results = []
    crossover_operators = [
        {"function": arithmetic_crossover, "label": "Arithmetic Crossover"},
        {"function": single_point_crossover, "label": "Single-Point Crossover"},
        {"function": uniform_crossover, "label": "Uniform Crossover"}
    ]

    for operator in crossover_operators:
        best_sol, best_dist, track_dist = genetic_algorithm(
            **common_params,
            selection_function=tournament_selection,
            crossover_function=operator["function"],
            mutation_function=gaussian_mutation,
            mutation_rate=0.1,
            mutation_param=0.1
        )
        crossover_results.append({
            "label": operator["label"],
            "distances": track_dist,
            "final_distance": best_dist
        })

    # ====================== MUTATION COMPARISON ======================
    mutation_results = []
    mutation_operators = [
        {"function": gaussian_mutation, "param": 0.1, "label": "Gaussian Mutation (σ=0.1)"},
        {"function": uniform_mutation, "param": 0.1, "label": "Uniform Mutation (±0.1)"}
    ]

    for operator in mutation_operators:
        best_sol, best_dist, track_dist = genetic_algorithm(
            **common_params,
            selection_function=tournament_selection,
            crossover_function=arithmetic_crossover,
            mutation_function=operator["function"],
            mutation_rate=0.1,
            mutation_param=operator["param"]
        )
        mutation_results.append({
            "label": operator["label"],
            "distances": track_dist,
            "final_distance": best_dist
        })

    # ====================== PLOT RESULTS ======================
    def plot_results(results, title):
        fig = go.Figure()
        for result in results:
            fig.add_trace(go.Scatter(
                x=list(range(1, maxgen)),
                y=result["distances"],
                name=f"{result['label']} (Final: {result['final_distance']:.2e})",
                mode="lines"
            ))
        fig.update_layout(
            title=title,
            xaxis_title="Generation",
            yaxis_title="Distance from Optimum (Log Scale)",
            yaxis_type="log",
            hovermode="x unified"
        )
        fig.show()

    plot_results(selection_results, "Selection Strategy Comparison")
    plot_results(crossover_results, "Crossover Operator Comparison")
    plot_results(mutation_results, "Mutation Operator Comparison")

# Run the comparison experiments
run_comparison_experiments()

## 📊 Genetic Algorithm Operator Comparison: Summary

### 🧪 Sphere Function (Easy, Convex)
- **Selection**:
  - **Tournament** performed best – strong pressure toward better individuals.
  - **Rank** slightly worse – stable but slower.
  - **Roulette** less consistent – sensitive to fitness scaling.

- **Crossover**:
  - **Arithmetic** led to fastest convergence – smooth offspring.
  - **Uniform** and **Single-Point** showed slower progress.

- **Mutation**:
  - **Gaussian (σ=0.1)** slightly outperformed uniform – better fine-tuning.
  - **Uniform (±0.1)** explored broader, but less focused.

### 🧪 Rosenbrock Function (Non-convex, Rugged)
- **Selection**:
  - **Tournament** again best – strong convergence on tough landscape.
  - **Rank** second best – slower but stable.
  - **Roulette** struggled – early convergence or stagnation.

- **Crossover**:
  - **Arithmetic** performed best – smooth transitions help this function.
  - **Single-Point** and **Uniform** were less efficient.

- **Mutation**:
  - **Gaussian** gave better local search – helped navigate narrow valleys.
  - **Uniform** less effective in refining precise solutions.

### ✅ Takeaways
- **Tournament + Arithmetic + Gaussian** is a solid default combo.
- Function complexity affects the best operator choice.
- Elitism helps maintain top solutions without losing diversity.
